# 04. Analysis & Figure Generation

## CS377 Reinforcement Learning — Team 1
### Concentration vs. Dispersion in Free-Setup Chess via AlphaZero

---

This notebook contains all **analysis and figure generation** code:

1. **Win Rate Analysis** — Dirichlet-multinomial posterior, credible intervals, additivity test
2. **Effective Piece Value Estimation** — Logistic regression (Q, R, B, N, P)
3. **Color Asymmetry Analysis** — First-move advantage interaction
4. **Comprehensive Analysis Script** — Full results analysis pipeline
5. **Figure Generation** — All paper figures (Fig 1-6)

---
## 1. Win Rate Analysis (Dirichlet-Multinomial)

Bayesian 추정 기반 승률 분석:
- **Posterior**: Dirichlet(α + W, α + D, α + L)
- **Score**: E[P(W)] + 0.5·E[P(D)]
- **Additivity Test**: P(score_i > score_j) pairwise comparison

Source: `handichess/analysis/winrate.py`

In [ ]:
"""
Win rate analysis with Dirichlet-multinomial posterior estimation.

Estimates handicap-side win rates per removal pattern, with credible
intervals, and tests whether the win rates are the same across patterns
(additivity test: "is 9 points the same however you remove them?").
"""

from __future__ import annotations

from typing import Optional

import numpy as np
import pandas as pd
from scipy import stats as sp_stats


def compute_win_rate(wins: int, draws: int, losses: int) -> float:
    """Compute the score (win rate) for the handicap side."""
    total = wins + draws + losses
    if total == 0:
        return 0.0
    return (wins + 0.5 * draws) / total


def dirichlet_multinomial_posterior(
    wins: int,
    draws: int,
    losses: int,
    prior_alpha: tuple[float, float, float] = (1.0, 1.0, 1.0),
    num_samples: int = 50_000,
    seed: int = 42,
) -> dict:
    """
    Bayesian estimation of win/draw/loss probabilities using
    a Dirichlet-multinomial model.

    Prior: Dirichlet(alpha) — default uniform (1,1,1).
    Posterior: Dirichlet(alpha + counts).

    Returns:
        Dict with: mean_score, ci_95, mean_probs, counts, total.
    """
    rng = np.random.default_rng(seed)

    alpha_post = np.array([
        prior_alpha[0] + wins,
        prior_alpha[1] + draws,
        prior_alpha[2] + losses,
    ])

    samples = rng.dirichlet(alpha_post, size=num_samples)
    pw, pd, pl = samples[:, 0], samples[:, 1], samples[:, 2]
    scores = pw + 0.5 * pd

    return {
        "mean_score": float(scores.mean()),
        "median_score": float(np.median(scores)),
        "ci_95": (float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))),
        "std_score": float(scores.std()),
        "mean_probs": {
            "win": float(pw.mean()),
            "draw": float(pd.mean()),
            "loss": float(pl.mean()),
        },
        "counts": {"wins": wins, "draws": draws, "losses": losses},
        "total": wins + draws + losses,
    }


def analyze_pattern_winrates(
    df: pd.DataFrame,
    prior_alpha: tuple[float, float, float] = (1.0, 1.0, 1.0),
) -> dict:
    """Analyze win rates for each pattern in the game log DataFrame."""
    results = {}
    for pid, group in df.groupby("pattern_id"):
        wins = (group["result"] == "win").sum()
        draws = (group["result"] == "draw").sum()
        losses = (group["result"] == "loss").sum()
        results[pid] = dirichlet_multinomial_posterior(
            int(wins), int(draws), int(losses), prior_alpha
        )
    return results


def additivity_test(
    pattern_results: dict,
    num_samples: int = 50_000,
    seed: int = 42,
) -> dict:
    """
    Test whether the handicap-side score is the same across all patterns.
    Method: Compare posterior score distributions pairwise.
    """
    rng = np.random.default_rng(seed)
    pattern_ids = sorted(pattern_results.keys())

    pattern_scores = {}
    for pid in pattern_ids:
        r = pattern_results[pid]
        c = r["counts"]
        alpha = np.array([1.0 + c["wins"], 1.0 + c["draws"], 1.0 + c["losses"]])
        samples = rng.dirichlet(alpha, size=num_samples)
        pattern_scores[pid] = samples[:, 0] + 0.5 * samples[:, 1]

    # Pairwise comparisons
    pairwise = {}
    for i, p1 in enumerate(pattern_ids):
        for p2 in pattern_ids[i + 1:]:
            prob_gt = float((pattern_scores[p1] > pattern_scores[p2]).mean())
            pairwise[f"{p1}_vs_{p2}"] = {
                f"P(score_{p1} > score_{p2})": prob_gt,
                f"P(score_{p2} > score_{p1})": 1.0 - prob_gt,
            }

    mean_scores = {pid: pattern_scores[pid].mean() for pid in pattern_ids}
    score_range = max(mean_scores.values()) - min(mean_scores.values())

    cis = {pid: (np.percentile(pattern_scores[pid], 2.5),
                 np.percentile(pattern_scores[pid], 97.5)) for pid in pattern_ids}

    homogeneous = True
    for i, p1 in enumerate(pattern_ids):
        for p2 in pattern_ids[i + 1:]:
            if cis[p1][1] < cis[p2][0] or cis[p2][1] < cis[p1][0]:
                homogeneous = False
                break

    return {
        "pairwise": pairwise,
        "mean_scores": mean_scores,
        "score_range": float(score_range),
        "homogeneous": homogeneous,
        "interpretation": (
            "Additivity holds: 9 points is equivalent regardless of removal pattern"
            if homogeneous else
            "Additivity violated: some removal patterns yield different effective handicaps"
        ),
    }


def print_winrate_summary(pattern_results: dict) -> None:
    """Pretty-print win rate results."""
    print("\n" + "=" * 70)
    print("Win Rate Analysis (Handicap Side)")
    print("=" * 70)
    print(f"{'Pattern':<25} {'Score':>7} {'95% CI':>18} {'W':>5} {'D':>5} {'L':>5} {'N':>5}")
    print("-" * 70)

    for pid in sorted(pattern_results.keys()):
        r = pattern_results[pid]
        ci = r["ci_95"]
        c = r["counts"]
        print(
            f"{pid:<25} {r['mean_score']:>7.3f} "
            f"[{ci[0]:.3f}, {ci[1]:.3f}] "
            f"{c['wins']:>5} {c['draws']:>5} {c['losses']:>5} "
            f"{r['total']:>5}"
        )
    print("=" * 70)

---
## 2. Effective Piece Value Estimation

로지스틱 회귀를 통해 기물의 실효 가치를 추정합니다:

$$\text{logit}\ P(\text{handicap wins}) = \beta_0 + \beta_Q \cdot \Delta Q + \beta_R \cdot \Delta R + \beta_B \cdot \Delta B + \beta_N \cdot \Delta N + \beta_P \cdot \Delta P + \beta_{\text{color}}$$

Source: `handichess/analysis/piece_values.py`

In [ ]:
"""
Effective piece value estimation via logistic regression.

Model:
  logit P(handicap side wins) = β₀ + βQ·ΔQ + βR·ΔR + βB·ΔB + βN·ΔN + βP·ΔP + β_color·color

β_piece estimates the effective value (in log-odds) of each piece type.
"""

from __future__ import annotations

from typing import Optional

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logit


STANDARD_VALUES = {"Q": 9, "R": 5, "B": 3, "N": 3, "P": 1}


def prepare_regression_data(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Prepare feature matrix X and target y from game log DataFrame."""
    records = []
    for _, row in df.iterrows():
        md = row.get("material_diff", {})
        if isinstance(md, str):
            import json
            md = json.loads(md)

        features = [
            1.0,  # intercept
            md.get("Q", 0),
            md.get("R", 0),
            md.get("B", 0),
            md.get("N", 0),
            md.get("P", 0),
            1.0 if row["handicap_side"] == "white" else 0.0,
        ]
        records.append((features, row["result_score"]))

    X = np.array([r[0] for r in records], dtype=np.float64)
    y = np.array([r[1] for r in records], dtype=np.float64)

    return X, y


def fit_logistic_regression(
    X: np.ndarray,
    y: np.ndarray,
    regularization: float = 0.01,
) -> dict:
    """
    Fit logistic regression to estimate effective piece values.
    Handles draws by treating them as y=0.5 (fractional outcome).
    """
    n_features = X.shape[1]

    def neg_log_likelihood(beta):
        logits = X @ beta
        logits = np.clip(logits, -20, 20)
        p = expit(logits)
        eps = 1e-10
        ll = y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)
        reg = regularization * np.sum(beta[1:] ** 2)
        return -np.sum(ll) + reg

    beta0 = np.zeros(n_features)
    result = minimize(neg_log_likelihood, beta0, method="L-BFGS-B")

    beta = result.x
    coeff_names = ["intercept", "ΔQ", "ΔR", "ΔB", "ΔN", "ΔP", "color"]
    coefficients = dict(zip(coeff_names, beta))

    piece_coeffs = {
        "Q": abs(beta[1]), "R": abs(beta[2]),
        "B": abs(beta[3]), "N": abs(beta[4]), "P": abs(beta[5]),
    }

    pawn_value = piece_coeffs["P"] if piece_coeffs["P"] > 0 else 1.0
    piece_values = {k: v / pawn_value for k, v in piece_coeffs.items()}

    comparison = {}
    for piece in STANDARD_VALUES:
        comparison[piece] = {
            "standard": STANDARD_VALUES[piece],
            "estimated": round(piece_values.get(piece, 0), 2),
            "ratio": round(piece_values.get(piece, 0) / STANDARD_VALUES[piece], 2)
            if STANDARD_VALUES[piece] > 0 else None,
        }

    return {
        "coefficients": coefficients,
        "piece_values_raw": piece_coeffs,
        "piece_values_normalized": piece_values,
        "standard_comparison": comparison,
        "color_effect": beta[6],
        "converged": result.success,
        "message": result.message,
    }


def analyze_piece_values(df: pd.DataFrame, **kwargs) -> dict:
    """End-to-end piece value analysis from a game log DataFrame."""
    X, y = prepare_regression_data(df)
    return fit_logistic_regression(X, y, **kwargs)


def print_piece_value_summary(results: dict) -> None:
    """Pretty-print piece value estimation results."""
    print("\n" + "=" * 60)
    print("Effective Piece Value Estimation")
    print("=" * 60)

    print("\nRaw Coefficients (log-odds):")
    for name, val in results["coefficients"].items():
        print(f"  {name:>12}: {val:+.4f}")

    print("\nNormalized Piece Values (Pawn = 1):")
    pv = results["piece_values_normalized"]
    print(f"  {'Piece':>6} {'Standard':>10} {'Estimated':>10} {'Ratio':>8}")
    print("  " + "-" * 36)
    for piece in ["Q", "R", "B", "N", "P"]:
        comp = results["standard_comparison"][piece]
        print(
            f"  {piece:>6} {comp['standard']:>10} "
            f"{comp['estimated']:>10.2f} {comp['ratio']:>8.2f}"
        )

    print(f"\nColor effect (white advantage): {results['color_effect']:+.4f}")
    print(f"Converged: {results['converged']}")
    print("=" * 60)

---
## 3. Color Asymmetry Analysis

선수(White)로 핸디캡을 받았을 때와 후수(Black)로 받았을 때의 차이를 분석합니다.

Source: `handichess/analysis/color_asym.py`

In [ ]:
"""
Black/white asymmetry analysis.

Measures the interaction between first-move advantage and material
handicap: does being handicapped as white vs. black make a difference?
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def analyze_color_asymmetry(df: pd.DataFrame) -> dict:
    """
    Analyze win rates broken down by handicap color.
    For each pattern, compare the handicap-side score when the
    handicapped player is white vs. black.
    """
    results = {}

    for pid, group in df.groupby("pattern_id"):
        pattern_result = {}

        for side in ["white", "black"]:
            side_games = group[group["handicap_side"] == side]
            wins = (side_games["result"] == "win").sum()
            draws = (side_games["result"] == "draw").sum()
            losses = (side_games["result"] == "loss").sum()

            pattern_result[side] = dirichlet_multinomial_posterior(
                int(wins), int(draws), int(losses)
            )

        w_score = pattern_result["white"]["mean_score"]
        b_score = pattern_result["black"]["mean_score"]
        pattern_result["delta"] = w_score - b_score
        pattern_result["interpretation"] = (
            "First-move advantage partially compensates handicap"
            if pattern_result["delta"] > 0.01
            else "First-move advantage does NOT compensate handicap"
            if pattern_result["delta"] < -0.01
            else "No significant color asymmetry"
        )

        results[pid] = pattern_result

    return results


def compute_aggregate_color_effect(df: pd.DataFrame) -> dict:
    """Compute the overall color effect across all patterns."""
    white_games = df[df["handicap_side"] == "white"]
    black_games = df[df["handicap_side"] == "black"]

    w_wins = (white_games["result"] == "win").sum()
    w_draws = (white_games["result"] == "draw").sum()
    w_losses = (white_games["result"] == "loss").sum()

    b_wins = (black_games["result"] == "win").sum()
    b_draws = (black_games["result"] == "draw").sum()
    b_losses = (black_games["result"] == "loss").sum()

    w_post = dirichlet_multinomial_posterior(int(w_wins), int(w_draws), int(w_losses))
    b_post = dirichlet_multinomial_posterior(int(b_wins), int(b_draws), int(b_losses))

    return {
        "white_handicap_score": w_post["mean_score"],
        "black_handicap_score": b_post["mean_score"],
        "delta": w_post["mean_score"] - b_post["mean_score"],
        "white_ci": w_post["ci_95"],
        "black_ci": b_post["ci_95"],
        "white_n": w_post["total"],
        "black_n": b_post["total"],
    }


def print_color_asymmetry_summary(results: dict) -> None:
    """Pretty-print color asymmetry results."""
    print("\n" + "=" * 75)
    print("Color Asymmetry Analysis (Handicap Side Score)")
    print("=" * 75)
    print(f"{'Pattern':<25} {'White-H':>9} {'Black-H':>9} {'Delta':>8} {'Interpretation'}")
    print("-" * 75)

    for pid in sorted(results.keys()):
        r = results[pid]
        w = r["white"]["mean_score"]
        b = r["black"]["mean_score"]
        delta = r["delta"]
        interp = r["interpretation"]
        print(f"{pid:<25} {w:>9.3f} {b:>9.3f} {delta:>+8.3f}  {interp}")

    print("=" * 75)

---
## 4. Comprehensive Analysis Script

Track A, Track B, Track B Stochastic 결과를 종합적으로 분석합니다.  
10개 섹션으로 구성: 전체 요약, 패턴별 결과, 손실된 Major/Minor 기물 수 기반 분석,  
색상 비대칭, 게임 길이, 종료 유형, Additivity 검정, 핵심 발견.

Source: `scripts/full_analysis.py`

In [ ]:
"""
Comprehensive analysis of all Track A, Track B, and Track B Stochastic results.
Produces summary statistics, win rates, concentration analysis, and color asymmetry.
"""
import json
from collections import defaultdict
from pathlib import Path

import numpy as np

# Pattern metadata
PATTERN_META = {
    "rook_bishop_pawn":    {"pieces": "R+B+P",   "points": "5+3+1=9", "num_removed": 3, "concentration": "high"},
    "rook_knight_pawn":    {"pieces": "R+N+P",   "points": "5+3+1=9", "num_removed": 3, "concentration": "high"},
    "bishop_bishop_knight":{"pieces": "B+B+N",   "points": "3+3+3=9", "num_removed": 3, "concentration": "high"},
    "rook_4pawns":         {"pieces": "R+4P",    "points": "5+4=9",   "num_removed": 5, "concentration": "medium"},
    "bishop_knight_3pawns":{"pieces": "B+N+3P",  "points": "3+3+3=9", "num_removed": 5, "concentration": "medium"},
    "bishop_6pawns":       {"pieces": "B+6P",    "points": "3+6=9",   "num_removed": 7, "concentration": "low"},
    "knight_6pawns":       {"pieces": "N+6P",    "points": "3+6=9",   "num_removed": 7, "concentration": "low"},
}

CONC_ORDER = ["high", "medium", "low"]


def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    games = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                games.append(json.loads(line))
    return games


def compute_stats(games):
    """Compute W/D/L counts and score from a list of game dicts."""
    wins = sum(1 for g in games if g["result"] == "win")
    draws = sum(1 for g in games if g["result"] == "draw")
    losses = sum(1 for g in games if g["result"] == "loss")
    total = len(games)
    score = (wins + 0.5 * draws) / total if total > 0 else 0
    return {
        "wins": wins, "draws": draws, "losses": losses,
        "total": total, "score": score
    }


def compute_ci(wins, draws, losses, n_samples=50000, seed=42):
    """Bayesian credible interval via Dirichlet-multinomial."""
    rng = np.random.default_rng(seed)
    alpha = np.array([1.0 + wins, 1.0 + draws, 1.0 + losses])
    samples = rng.dirichlet(alpha, size=n_samples)
    scores = samples[:, 0] + 0.5 * samples[:, 1]
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def analyze_by_color(games):
    """Split games by noq_side and compute stats."""
    by_noq_color = defaultdict(list)
    for g in games:
        by_noq_color[g["noq_side"]].append(g)
    
    result = {}
    for color in ["white", "black"]:
        color_games = by_noq_color.get(color, [])
        if color_games:
            stats = compute_stats(color_games)
            noq_wins = stats["losses"]
            noq_draws = stats["draws"]
            noq_losses = stats["wins"]
            noq_total = stats["total"]
            noq_score = (noq_wins + 0.5 * noq_draws) / noq_total if noq_total > 0 else 0
            result[color] = {
                "noq_wins": noq_wins, "noq_draws": noq_draws, "noq_losses": noq_losses,
                "total": noq_total, "noq_score": noq_score
            }
    return result


# ── Usage: run full analysis on JSONL files ──
# results_dir = Path("runs/results")
# patterns = list(PATTERN_META.keys())
# See scripts/full_analysis.py for the complete 10-section analysis pipeline.

print("See scripts/full_analysis.py for the full comprehensive analysis pipeline.")
print("This generates: Overall Summary, Per-Pattern Results, Concentration Analysis,")
print("Color Asymmetry, Game Length, Termination Types, Additivity Test, Key Findings.")

---
## 5. Figure Generation (Paper Figures)

논문에 사용되는 6개의 Figure를 생성합니다:

| Figure | Description |
|--------|-------------|
| Fig 1 | Game Outcomes by Handicap Pattern (Stacked horizontal bar) |
| Fig 2 | Q-Score by Minor Pieces Lost (Bar chart) |
| Fig 3 | Q-Score by Major vs Minor Pieces Lost (Heatmap) |
| Fig 4 | Pairwise Bayesian Comparison (Heatmap) |
| Fig 5 | Color Asymmetry (Grouped bar chart) |
| Fig 6 | ScratchZero vs Lc0 Comparison (Grouped bar with CI) |

Source: `scripts/generate_figures.py`

In [ ]:
"""
Generate all paper figures from experiment results.
Outputs: figures/fig1_outcomes.pdf ... fig6_track_comparison.pdf
"""
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

# ── Style ──
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans", "DejaVu Sans", "sans-serif"],
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# ── Data ──
PATTERNS_ORDER = [
    "bishop_bishop_knight",
    "bishop_knight_3pawns",
    "rook_bishop_pawn",
    "rook_knight_pawn",
    "bishop_6pawns",
    "knight_6pawns",
    "rook_4pawns",
]

LABELS = {
    "rook_bishop_pawn":     "R+B+P",
    "rook_knight_pawn":     "R+N+P",
    "bishop_bishop_knight": "B+B+N",
    "rook_4pawns":          "R+4P",
    "bishop_knight_3pawns": "B+N+3P",
    "bishop_6pawns":        "B+6P",
    "knight_6pawns":        "N+6P",
}

CONC = {
    "rook_bishop_pawn": "High", "rook_knight_pawn": "High",
    "bishop_bishop_knight": "High",
    "rook_4pawns": "Medium", "bishop_knight_3pawns": "Medium",
    "bishop_6pawns": "Low", "knight_6pawns": "Low",
}

CONC_COLOR = {"High": "#4A90D9", "Medium": "#F5A623", "Low": "#D0021B"}


def wdl(games):
    w = sum(1 for g in games if g["result"] == "win")
    d = sum(1 for g in games if g["result"] == "draw")
    l = sum(1 for g in games if g["result"] == "loss")
    return w, d, l


def score(w, d, l):
    t = w + d + l
    return (w + 0.5 * d) / t if t > 0 else 0


def ci95(w, d, l, n=50000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.array([1.0+w, 1.0+d, 1.0+l])
    s = rng.dirichlet(a, size=n)
    sc = s[:,0] + 0.5*s[:,1]
    return float(np.percentile(sc, 2.5)), float(np.percentile(sc, 97.5))

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  FIGURE 1: Stacked Horizontal Bar — Game Outcomes
# ═══════════════════════════════════════════════════════════════

def fig1(data):
    fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
    
    colors_w = "#3FA055"   # Green
    colors_d = "#FFFACD"   # Light yellow
    colors_l = "#DB4C37"   # Red-orange
    
    labels = []
    w_counts, d_counts, l_counts = [], [], []
    
    for pat in PATTERNS_ORDER:
        if pat not in data: continue
        w, d, l = wdl(data[pat])
        labels.append(LABELS[pat])
        w_counts.append(w)
        d_counts.append(d)
        l_counts.append(l)
    
    y = np.arange(len(labels))
    
    ax.barh(y, w_counts, color=colors_w, height=0.5, label="Q-side wins")
    ax.barh(y, d_counts, left=w_counts, color=colors_d, height=0.5, label="Draw")
    left2 = [w+d for w, d in zip(w_counts, d_counts)]
    ax.barh(y, l_counts, left=left2, color=colors_l, height=0.5, label="Q-side loss")
    
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xticks([])
    ax.invert_yaxis()
    
    for idx in range(len(labels)):
        w, d, l = w_counts[idx], d_counts[idx], l_counts[idx]
        if w > 0:
            ax.text(w/2, y[idx], str(w), color="white", ha="center", va="center", fontsize=11)
        if d > 0:
            ax.text(w + d/2, y[idx], str(d), color="gray", ha="center", va="center", fontsize=11)
        if l > 0:
            ax.text(w + d + l/2, y[idx], str(l), color="white", ha="center", va="center", fontsize=11)

    handles, labels_leg = ax.get_legend_handles_labels()
    fig.legend(handles, labels_leg, loc="upper center", ncol=3, fontsize=13, bbox_to_anchor=(0.5, 1.1), frameon=True)
    fig.text(0.5, -0.05, "Figure 1: Game Outcomes by Handicap Pattern (Lc0)", ha="center", fontsize=15, fontweight="bold")
    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════
#  FIGURE 2: Q-Score by Number of Minor Pieces Lost
# ═══════════════════════════════════════════════════════════════

def fig2(data):
    fig, ax = plt.subplots(figsize=(7, 5))
    
    MINOR_GROUPS = {
        "0 Minors": ["rook_4pawns"],
        "1 Minor": ["rook_knight_pawn", "rook_bishop_pawn", "bishop_6pawns", "knight_6pawns"],
        "2 Minors": ["bishop_knight_3pawns"],
        "3 Minors": ["bishop_bishop_knight"]
    }
    group_names = list(MINOR_GROUPS.keys())
    scores, lo_errs, hi_errs = [], [], []
    
    for gn in group_names:
        pats = MINOR_GROUPS[gn]
        if len(pats) > 1:
            pattern_scores = []
            for p in pats:
                if p in data:
                    w, d, l = wdl(data[p])
                    pattern_scores.append(score(w, d, l))
            if len(pattern_scores) > 0:
                s = np.mean(pattern_scores)
                lo_err = s - np.min(pattern_scores)
                hi_err = np.max(pattern_scores) - s
            else:
                s, lo_err, hi_err = 0, 0, 0
        else:
            p = pats[0]
            if p in data:
                w, d, l = wdl(data[p])
                s = score(w, d, l)
                lo, hi = ci95(w, d, l)
                lo_err = s - lo
                hi_err = hi - s
            else:
                s, lo_err, hi_err = 0, 0, 0

        scores.append(s)
        lo_errs.append(lo_err)
        hi_errs.append(hi_err)

    x = np.arange(len(group_names))
    width = 0.5
    
    bars = ax.bar(x, scores, width, 
                  yerr=[lo_errs, hi_errs], capsize=5,
                  color="#ED7D31", edgecolor="black", linewidth=0.8, alpha=0.85, zorder=2,
                  error_kw={'elinewidth': 1.5, 'capthick': 1.5, 'zorder': 3})
                
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.5, zorder=1)
    ax.set_xticks(x)
    ax.set_xticklabels(group_names, fontsize=12)
    ax.set_ylabel("Q-Score", fontsize=13)
    ax.set_ylim(0.0, 1.05)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    
    for i, bar in enumerate(bars):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + hi_errs[i] + 0.02, f"{h:.3f}",
               ha="center", va="bottom", fontsize=16, fontweight="bold")
    
    ax.set_title("Figure 2: Q-Score by Number of Minor Pieces Lost (Lc0)", fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════
#  FIGURE 3: Q-Score by Major vs Minor Pieces Lost
# ═══════════════════════════════════════════════════════════════

def fig3_heatmap(data):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    DATA_MAP = {
        (1, 0): ["rook_4pawns"],
        (0, 1): ["bishop_6pawns", "knight_6pawns"],
        (1, 1): ["rook_bishop_pawn", "rook_knight_pawn"],
        (0, 2): ["bishop_knight_3pawns"],
        (0, 3): ["bishop_bishop_knight"]
    }
    
    matrix = np.full((2, 4), np.nan)
    text_matrix = np.empty((2, 4), dtype=object)
    
    for r in [0, 1]:
        for m in [0, 1, 2, 3]:
            if (r, m) in DATA_MAP:
                pats = DATA_MAP[(r, m)]
                sg = []
                for p in pats:
                    if p in data:
                        sg.extend(data[p])
                if len(sg) > 0:
                    w, d, l = wdl(sg)
                    s = score(w, d, l)
                    row = 1 - r
                    col = m
                    matrix[row, col] = s
                    text_matrix[row, col] = f"{s:.3f}"
    
    cmap = plt.get_cmap("OrRd").copy()
    cmap.set_bad(color="white")
    im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["1 Rook", "0 Rooks"], fontsize=12)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(["0 Minors", "1 Minor", "2 Minors", "3 Minors"], fontsize=12)
    
    ax.set_ylabel("Major Pieces Lost", fontsize=13)
    ax.set_xlabel("Minor Pieces Lost", fontsize=13)
    
    for r in range(2):
        for c in range(4):
            if not np.isnan(matrix[r, c]):
                val = matrix[r, c]
                color = "white" if val > 0.6 else "black"
                ax.text(c, r, text_matrix[r, c], ha="center", va="center", color=color, fontsize=17, fontweight="bold")
            else:
                ax.text(c, r, "N/A", ha="center", va="center", color="gray", fontsize=14, fontstyle="italic")
    
    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 2, 1), minor=True)
    ax.grid(which="minor", color="black", linestyle='-', linewidth=2)
    ax.tick_params(which="minor", size=0)

    # Thick outer borders
    for spine in ax.spines.values():
        spine.set_linewidth(2)
        spine.set_edgecolor('black')



    ax.set_title("Figure 3: Q-Score by Major vs Minor Pieces Lost (Lc0)", fontsize=14, fontweight="bold", y=1.03)
    fig.tight_layout()
    return fig


#  FIGURE 4: Pairwise Bayesian Heatmap
# ═══════════════════════════════════════════════════════════════

def fig_heatmap(data):
    avail = [p for p in PATTERNS_ORDER if p in data]
    n = len(avail)
    
    rng = np.random.default_rng(42)
    samples = {}
    for p in avail:
        w, d, l = wdl(data[p])
        s = rng.dirichlet([1+w, 1+d, 1+l], 50000)
        samples[p] = s[:,0] + 0.5*s[:,1]
    
    matrix = np.full((n, n), np.nan)
    for i, p1 in enumerate(avail):
        for j, p2 in enumerate(avail):
            if i != j:
                matrix[i, j] = float((samples[p1] > samples[p2]).mean())
    
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1, aspect="equal")
    
    tick_labels = [LABELS[p] for p in avail]
    ax.set_xticks(range(n))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(n))
    ax.set_yticklabels(tick_labels)

    for i in range(n):
        for j in range(n):
            if i != j:
                ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=11, color="white")
    
    fig.text(0.5, -0.05, "Figure 4: Pairwise Bayesian Comparison (Lc0)", ha="center", fontsize=15, fontweight="bold")
    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════
#  FIGURE 5: Color Asymmetry
# ═══════════════════════════════════════════════════════════════

def fig_color(data):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    patterns = [p for p in PATTERNS_ORDER if p in data]
    labels = [LABELS[p] for p in patterns]
    
    w_scores, b_scores = [], []
    for pat in patterns:
        wg = [g for g in data[pat] if g["noq_side"] == "black"]
        bg = [g for g in data[pat] if g["noq_side"] == "white"]
        qw, qd, ql = wdl(wg)
        bw, bd, bl = wdl(bg)
        w_scores.append(score(qw, qd, ql))
        b_scores.append(score(bw, bd, bl))
    
    x = np.arange(len(patterns))
    width = 0.35
    
    ax.bar(x - width/2, w_scores, width, color="#5B9BD5", edgecolor="black", linewidth=0.8,
           label="Queen = White (has first-move)", zorder=3)
    ax.bar(x + width/2, b_scores, width, color="#333333", edgecolor="black", linewidth=0.8,
           label="Queen = Black", zorder=3)
    
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.5, zorder=1)
    
    for i in range(len(patterns)):
        delta = w_scores[i] - b_scores[i]
        mid = max(w_scores[i], b_scores[i]) + 0.03
        ax.annotate(f"Δ={delta:+.2f}", xy=(x[i], mid), ha="center", fontsize=10)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Q-Score (Queen's Win Rate)")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    
    fig.text(0.5, -0.05, "Figure 5: Color Asymmetry (Lc0)", ha="center", fontsize=15, fontweight="bold")
    fig.tight_layout()
    return fig


# ═══════════════════════════════════════════════════════════════
#  FIGURE 6: Track A vs Track B Comparison
# ═══════════════════════════════════════════════════════════════

def fig_comparison(track_a_data, track_bs_data):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    patterns = [p for p in PATTERNS_ORDER if p in track_a_data and p in track_bs_data]
    labels = [LABELS[p] for p in patterns]
    
    a_scores, bs_scores = [], []
    a_ci_lo, a_ci_hi = [], []
    bs_ci_lo, bs_ci_hi = [], []
    
    for pat in patterns:
        w, d, l = wdl(track_a_data[pat])
        s = score(w, d, l)
        lo, hi = ci95(w, d, l)
        a_scores.append(s); a_ci_lo.append(s-lo); a_ci_hi.append(hi-s)
        
        w, d, l = wdl(track_bs_data[pat])
        s = score(w, d, l)
        lo, hi = ci95(w, d, l)
        bs_scores.append(s); bs_ci_lo.append(s-lo); bs_ci_hi.append(hi-s)
    
    x = np.arange(len(patterns))
    width = 0.35
    
    ax.bar(x - width/2, a_scores, width,
           yerr=[a_ci_lo, a_ci_hi], capsize=4,
           color="#AAAAAA", edgecolor="black", linewidth=0.8,
           label="ScratchZero (~500 Elo)", zorder=3)
    ax.bar(x + width/2, bs_scores, width,
           yerr=[bs_ci_lo, bs_ci_hi], capsize=4,
           color="#2E86AB", edgecolor="black", linewidth=0.8,
           label="Lc0 (~2200+ Elo)", zorder=3)
    
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.5, zorder=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Q-Score")
    ax.legend(loc="upper left")
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    
    fig.text(0.5, -0.05, "Figure 6: ScratchZero vs Lc0 Comparison", ha="center", fontsize=15, fontweight="bold")
    fig.tight_layout()
    return fig


print("All figure generation functions defined.")
print("To generate figures, load JSONL data and call: fig1(data), fig2(data), ...")
print("See scripts/generate_figures.py for the complete pipeline.")
